In [1]:
import torch
import datasets
import nltk
import glob
import re
import collections
import numpy as np
import torch.nn as nn
import time
import torch.nn.functional as F
from datasets import load_dataset, Dataset, DatasetDict
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from nltk.tokenize import wordpunct_tokenize
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
import os

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /Users/admin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/admin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Data Preparing

In [3]:
def load_and_split_data(data_path):
    data = []
    labels = []

    for label, label_val in [("pos", 1), ("neg", 0)]:
        dir_path = f"{data_path}/{label}"
        files = glob.glob(f"{dir_path}/*.txt")
        for file in files:
            with open(file, 'r', encoding='utf-8') as f:
                text = f.read().strip()
            data.append(text)
            labels.append(label_val)

    dataset = datasets.Dataset.from_dict({'text': data, 'label': labels})
    
    #split the dataset: train:80%, test:10%, val:10%
    train_test_split = dataset.train_test_split(test_size=0.2, seed=42)
    test_val_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

    train_data = train_test_split['train']
    test_data = test_val_split['test']
    val_data = test_val_split['train']

    splited_dataset = datasets.DatasetDict({"train": train_data, "test": test_data, "validation": val_data})

    return splited_dataset

In [4]:
root_dir = os.getenv("IMDB_TRAIN_DIR", "data/aclImdb/train")
dataset = load_and_split_data(root_dir)

print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2500
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2500
    })
})
{'text': 'I can sit through this movie once, but I doubt I could make it through a second time. Mildly entertaining mainly for the physical presence of Lindsay Lohan. The fun of Matt Dillon(think Something About Mary), and the re-emergence of a more serious acting Micheal Keaton. This is not the fun romp The Love Bug was but it is watchable. One of my main detractions from the movie was the fact that Herbie had evolved into R2D2 antics for the most part. I was bored with the headlight eyes effects about the second time of the 30 or so times they were used and the bending front fender caused the same reaction from me. Go see this with your little ones, i.e. the "single digit bracket" kids and don\'t expect

In [5]:
def preprocess_text(text):
    ''''
    clean the raw data:
    removing the html tags and convert to lowercase,
    tokenizing the text into words.
    
    '''
    filterwords = nltk.corpus.stopwords.words('english')

    # Remove HTML tags
    text = BeautifulSoup(text, 'html.parser').get_text()

    #convert to lowercase
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  
    
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in filterwords]

    return tokens

In [6]:
dataset = dataset.map(lambda example: {'tokens': preprocess_text(example['text'])}, batched=False, num_proc=1)

print(dataset['train'][0]['tokens'])

Map (num_proc=1):   0%|          | 0/20000 [00:00<?, ? examples/s]

/var/folders/2q/cw244_nn68x_x0h83_9wl_980000gn/T/ipykernel_13447/69977204.py:11: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, 'html.parser').get_text()


Map (num_proc=1):   0%|          | 0/2500 [00:00<?, ? examples/s]

/var/folders/2q/cw244_nn68x_x0h83_9wl_980000gn/T/ipykernel_13447/69977204.py:11: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, 'html.parser').get_text()


Map (num_proc=1):   0%|          | 0/2500 [00:00<?, ? examples/s]

/var/folders/2q/cw244_nn68x_x0h83_9wl_980000gn/T/ipykernel_13447/69977204.py:11: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, 'html.parser').get_text()


['sit', 'movie', 'doubt', 'could', 'make', 'second', 'time', 'mildly', 'entertaining', 'mainly', 'physical', 'presence', 'lindsay', 'lohan', 'fun', 'matt', 'dillonthink', 'something', 'mary', 'reemergence', 'serious', 'acting', 'micheal', 'keaton', 'fun', 'romp', 'love', 'bug', 'watchable', 'one', 'main', 'detractions', 'movie', 'fact', 'herbie', 'evolved', 'rd', 'antics', 'part', 'bored', 'headlight', 'eyes', 'effects', 'second', 'time', 'times', 'used', 'bending', 'front', 'fender', 'caused', 'reaction', 'go', 'see', 'little', 'ones', 'ie', 'single', 'digit', 'bracket', 'kids', 'dont', 'expect', 'lot', 'filmx']


In [7]:
def build_vocab(token_list, max_size=10000):
    all_tokens = []
    stoi = {}
    itos = {}

    for tokens in token_list:
        for token in tokens:
            all_tokens.append(token)

    counter = collections.Counter(all_tokens)
    
    most_common = counter.most_common(max_size - 2) # Reserve 2 spots for <PAD> and <UNK>
    
    vocab_tokens = ['<pad>', '<unk>'] + [token for token, _ in most_common]

    for idx, token in enumerate(vocab_tokens):
        stoi[token] = idx
        itos[idx] = token

    return stoi, itos, len(stoi) #vocab size

In [8]:
# build a vocabulary from the training data
stoi, itos, vocab_size = build_vocab(dataset['train']['tokens'])

print(f"Vocabulary size: {vocab_size}")
print(list(stoi.items())[:5]) 

print(dataset['train']['tokens'])

Vocabulary size: 10000
[('<pad>', 0), ('<unk>', 1), ('movie', 2), ('film', 3), ('one', 4)]
Column([['sit', 'movie', 'doubt', 'could', 'make', 'second', 'time', 'mildly', 'entertaining', 'mainly', 'physical', 'presence', 'lindsay', 'lohan', 'fun', 'matt', 'dillonthink', 'something', 'mary', 'reemergence', 'serious', 'acting', 'micheal', 'keaton', 'fun', 'romp', 'love', 'bug', 'watchable', 'one', 'main', 'detractions', 'movie', 'fact', 'herbie', 'evolved', 'rd', 'antics', 'part', 'bored', 'headlight', 'eyes', 'effects', 'second', 'time', 'times', 'used', 'bending', 'front', 'fender', 'caused', 'reaction', 'go', 'see', 'little', 'ones', 'ie', 'single', 'digit', 'bracket', 'kids', 'dont', 'expect', 'lot', 'filmx'], ['im', 'surprised', 'many', 'female', 'voters', 'even', 'give', 'film', 'better', 'marks', 'thought', 'film', 'target', 'audience', 'adult', 'male', 'whipped', 'tortured', 'women', 'merciless', 'revenge', 'high', 'body', 'count', 'typical', 'ingredients', 'introduced', 'film', '

In [9]:
def load_glove_embeddings(glove_file, stoi, embedding_dim=300):
    embedding = np.random.normal(size=(len(stoi), embedding_dim)).astype(np.float32)
    with open(glove_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            if word in stoi:
                embedding[stoi[word]] = np.array(parts[1:], dtype=np.float32)
    
    return torch.tensor(embedding) 

In [10]:
glove_file = os.getenv("GLOVE_PATH", "data/glove/glove.6B.300d.txt")
pretrained_embeddings = load_glove_embeddings(glove_file, stoi, embedding_dim=300)

In [11]:
def vectorize(tokens, stoi):
    integer_sequence = []
    for token in tokens:
        integer_sequence.append(stoi.get(token, stoi['<unk>']))

    return integer_sequence

In [12]:
dataset = dataset.map(lambda example: {'integer_sequence': vectorize(example['tokens'], stoi)}, batched=False)

print(dataset['train'][90]['integer_sequence'][:10])
print(dataset) 

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

[722, 666, 75, 3969, 1, 372, 255, 777, 1252, 1533]
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'tokens', 'integer_sequence'],
        num_rows: 20000
    })
    test: Dataset({
        features: ['text', 'label', 'tokens', 'integer_sequence'],
        num_rows: 2500
    })
    validation: Dataset({
        features: ['text', 'label', 'tokens', 'integer_sequence'],
        num_rows: 2500
    })
})


In [13]:
def collate_fn(batch):
    sequences = [torch.tensor(item['integer_sequence']) for item in batch]
    labels = torch.tensor([item['label'] for item in batch])

    #handle sequences of varying lengths by implementing padding
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=stoi['<pad>'])
    return padded_sequences, labels

In [14]:
batch_size = 32
train_loader = DataLoader(dataset['train'], batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(dataset['validation'], batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(dataset['test'], batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

for inputs, labels in train_loader:
    print("Input batch shape:", inputs.shape)
    print("Labels batch shape:", labels.shape)
    break

Input batch shape: torch.Size([32, 491])
Labels batch shape: torch.Size([32])


## Model definition

In [15]:
class LSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim
                 , dropout_rate=0.5, embedding_type='random', pretrained_embedding=None, 
                 freeze_embedding=False):
        super().__init__()
        self.num_class = 2
        self.embedding_type = embedding_type

        # randomly initialized embedding layer that is trained from scratch
        if self.embedding_type == 'random':
            self.embedding = nn.Embedding(vocab_size, embedding_dim)

        #embedding layer initialized with pre-trained word embeddings
        elif self.embedding_type == 'pretrained':
            self.embedding = nn.Embedding.from_pretrained(pretrained_embedding, freeze=freeze_embedding)
        
        self.lstm = nn.LSTM(input_size=embedding_dim,
                            hidden_size=hidden_dim,
                            num_layers=1,
                            bidirectional=False,
                            dropout=0.0,
                            batch_first=True
                            )
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, inputs):
        embedded = self.embedding(inputs)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        hidden = self.dropout(hidden[-1])
        output = self.fc(hidden)
        return output



## Train the Model

In [16]:
def train_model(model, train_loader, val_loader, criterion=nn.CrossEntropyLoss(), num_epochs=10, optimizer=None):
    model.to(device)
    model.train()

    #keep track of training loss for ploting
    all_train_losses = []
    all_val_losses = []
    optimizer = optimizer

    print(f"training on dataset with n = {len(train_loader.dataset)}")

    for epoch in range(1, num_epochs + 1):
        model.train()
        start = time.time()
        total_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
        
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        all_train_losses.append(avg_train_loss)

    #evaluate the model on validation set after training
    
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for val_inputs, val_labels in val_loader:
                val_inputs, val_labels = val_inputs.to(device), val_labels.to(device)
                val_outputs = model(val_inputs)
                val_loss = criterion(val_outputs, val_labels)
                total_val_loss += val_loss.item()

            avg_val_loss = total_val_loss / len(val_loader)
            all_val_losses.append(avg_val_loss)

        print(f"Epoch {epoch}: avg train loss = {avg_train_loss:.3e}, avg val loss = {avg_val_loss:.3e}; time: {time.time() - start:.1f}s")


    return all_train_losses, all_val_losses 

# Test the Model

In [17]:
def test_model(model, test_loader):
    model = model.to(device)
    model.eval()

    all_predictions = []
    all_true = []
    all_probs = []

    with torch.no_grad():

        for inputs, true_labels in test_loader:

            inputs = inputs.to(device)
            true_labels =  true_labels.to(device)

            outputs = model(inputs)
            predicted = torch.argmax(outputs, dim=1)

            prob_result = torch.softmax(outputs, dim=1)
            pos_prob = prob_result[:, 1]

            predictions_cpu = predicted.cpu()
            true_labels_cpu =  true_labels.cpu()
            prob_cpu = pos_prob.cpu()

            predictions_numpy = predictions_cpu.numpy()
            true_numpy =  true_labels_cpu.numpy()
            prob_np = prob_cpu.numpy()

            for i in range(len(predictions_numpy)):
                all_predictions.append(predictions_numpy[i])
            for i in range(len( true_numpy)):
                all_true.append( true_numpy[i])
            for i in range(len(prob_np)):
                all_probs.append(prob_np[i])

    # 1. Accuracy:
    correct_count = 0
    total_count = len(all_predictions)

    for i in range(total_count):
        if all_predictions[i] == all_true[i]:
            correct_count += 1

    accuracy = correct_count / total_count

    # 2. Precision
    true_positive = 0
    false_positive = 0

    for i in range(total_count):
        if all_predictions[i] == 1:
            if all_true[i] == 1:
                true_positive += 1
            else:
                false_positive += 1

    if (true_positive + false_positive) > 0:
        precision = true_positive / (true_positive + false_positive)
    else:
        precision = 0

    # 3. Recall
    actual_positive = 0

    for i in range(total_count):
        if all_true[i] == 1:
            actual_positive += 1

    if actual_positive > 0:
        recall = true_positive / actual_positive
    else:
        recall = 0

    # 4. F1 Score
    if (precision + recall) > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0

    # 5. Confusion Matrix
    true_negative = 0
    false_negative = 0

    for i in range(total_count):
        if all_predictions[i] == 0:
            if all_true[i] == 0:
                true_negative += 1
            else:
                false_negative += 1

    confusion_matrix_result = [
        [true_negative, false_positive],
        [false_negative, true_positive]
    ]

    print("\nTest Results:")
    print(f"Correct predictions: {correct_count}")
    print(f"Total predictions: {total_count}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    print("\nConfusion Matrix:")
    print("        Predicted Negative  Predicted Positive")
    print(f"Actual Negative   {true_negative}               {false_positive}")
    print(f"Actual Positive   {false_negative}               {true_positive}")

    print("\nSimple Evaluation:")
    if accuracy > 0.8:
        print("Very Good!")
    elif accuracy > 0.6:
        print("Not Bad")
    else:
        print("Not Good")

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    right_num = correct_count
    wrong_num = total_count - correct_count
    pie_data = [right_num, wrong_num]
    pie_labels = [f'correct\n{right_num}', f'wrong\n{wrong_num}']
    pie_colors = ['lightgreen', 'lightcoral']

    plt.pie(pie_data, labels=pie_labels, colors=pie_colors, autopct='%1.1f%%')
    plt.title('Accuracy Pie Chart')

    plt.subplot(1, 2, 2)


    fpr, tpr, thresholds = roc_curve(all_true, all_probs)
    auc_score = auc(fpr, tpr)

    plt.plot(fpr, tpr, color='blue', linewidth=2, label=f'AUC = {auc_score:.3f}')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=1, label='Random Classifier')
    plt.xlabel('False positive rate')
    plt.ylabel('True positive rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"AUC: {auc_score:.4f}")
    if auc_score > 0.9:
        print("Excellent")
    elif auc_score > 0.8:
        print("Not bad")
    else:
        print("Not Good")

    return accuracy, f1, auc_score

In [ ]:
#initialize the model 
    
#randomly initialized embedding
model_random = LSTM(vocab_size=len(stoi), embedding_dim=300, hidden_dim=512, output_dim=2, embedding_type='random')
#pre-trained embedding(frozen)
model_pretrained_frozen = LSTM(vocab_size=len(stoi), embedding_dim=300, hidden_dim=512, output_dim=2,
                            embedding_type='pretrained', pretrained_embedding=pretrained_embeddings, freeze_embedding=True)

optimizer_random = torch.optim.Adam(model_random.parameters(), lr=0.0008)
optimizer_pretrained_frozen = torch.optim.Adam(model_pretrained_frozen.parameters(), lr=0.0008)

train_model(model_random, train_loader, val_loader, optimizer=optimizer_random)

training on dataset with n = 20000


In [ ]:
train_model(model_pretrained_frozen, train_loader, val_loader, optimizer=optimizer_pretrained_frozen)

In [ ]:
print("Testing random embedding model:")
acc1, f1_1, auc1 = test_model(model_random, test_loader)

print("Testing pretrained embedding model:")
acc2, f1_2, auc2 = test_model(model_pretrained_frozen, test_loader)

print("\nModel comparison:")
print("Metric        Random    Pretrained  Diff")
print("-----------------------------------------")
print(f"Accuracy      {acc1:.4f}     {acc2:.4f}     {acc2-acc1:+.4f}")
print(f"F1            {f1_1:.4f}     {f1_2:.4f}     {f1_2-f1_1:+.4f}")
print(f"AUC           {auc1:.4f}     {auc2:.4f}     {auc2-auc1:+.4f}")

if acc2 > acc1 and f1_2 > f1_1 and auc2 > auc1:
    print("Pretrained embedding is better")
elif acc1 > acc2 and f1_1 > f1_2 and auc1 > auc2:
    print("Random embedding is better")
else:
    print("Mixed results")